# Lab 3, part A: the policy

Lab 3 (A+B) costs: $0.134

**This part builds the half of a support agent that does not ask the model to co-operate.**

Scenario 1, a customer support resolution agent. It handles returns, billing disputes and
account questions against four backend tools, and it is measured on first-contact
resolution: resolve what it can, escalate what it must, and be right about which is which.

The exam's evidence for why this lab exists is one number. Told in its system prompt to
verify the customer first, a support agent skipped `get_customer` in **12% of production
cases** and looked orders up on the name the customer typed, which misidentified accounts
and refunded the wrong people. Stronger wording moves that number. It never reaches zero,
because it still asks the model to choose to comply.

So part A builds the half that does not ask: four tools, the failure shapes they return,
two gates that decide what may run, one pass that decides what the model gets to read, and
a handoff a human can act on without the transcript.

**No API calls here.** Every mechanism is a plain async function, so all of it is proven by
calling it, offline and free. Part B hands the lot to an agent and watches what changes.

## 1. The workspace

The tools read three JSON tables and a policy file. `fixtures.py` beside this notebook
writes them, and it is worth knowing two things it plants there.

Two customers share the name **Sam Okafor**, which is what makes verification ambiguous.
And the policy is silent on competitor price matching. That silence is deliberate, and
part B's third conversation turns on it.

![Lab 3: customer support agent](../diagrams/lab-03-customer-support-agent.png)


In [1]:
import json

import labkit

lab = labkit.start(model_env=None, credential="none")

import fixtures                    # the data, so it is not 110 lines of this notebook
import support.hooks as H          # the policy around the tools
import support.tools as T          # the four tools themselves

fixtures.build(lab.workspace)

workspace  /Users/pizady/Code/ultimate-ccar-f-labs/lab_03/workspace
6 customers, two of them sharing a name
5 orders, 33 fields each, 2 invoices
policy.md: 30 day window, own site only, 500 pound agent limit, and nothing at all about competitors


## 2. Four tools, and what each returns when it fails

**Half of a tool's job is answering. The other half is failing in a shape the agent can act
on.**

The four the scenario names run in-process, inside this Python session, through the Agent
SDK's own MCP server. No subprocess, no transport, no `.mcp.json`. The key they are
registered under is what names them, so `support` gives us `mcp__support__get_customer` and
its three siblings: the strings every rule in this lab keys on.

| Category | Retryable | What the agent should do | Here |
|---|---|---|---|
| transient | yes | retry, with a delay | the orders API times out |
| validation | yes, after the input changes | fix the arguments, then retry | no account matches that identifier |
| business | no | explain it to the person, in their terms | the order is outside the refund window |
| permission | no | escalate | the agent is not authorised |

**The fifth shape is not a failure at all.** A query that ran correctly and matched nothing,
or matched more than one thing, is a **success**. `get_customer` finding two accounts is the
answer, not an error. Hiding that behind a ranking that returns the likeliest match is the
exact failure this lab is built to avoid: the agent must ask for another identifier, and it
can only do that if the tool tells it the truth.

**One spelling note**, because both appear in the course. On the MCP wire the flag is
`isError`, which is the exam's term. The SDK's `@tool` decorator forwards `is_error`, so
that is what these tools return. Lab 1's FastMCP tools had to **raise**, because a returned
dictionary was reported as a success; an SDK tool returns the flag instead.

Below: the failure contract, defined once, and the one tool whose description has to carry
the ambiguity rule. The other three are in `support/tools.py`.

In [2]:
labkit.show_source(T, "failure", "get_customer")

def failure(category, message, retryable, **extra):
    """A failure the agent can act on, rather than one it has to guess about."""
    payload = {"errorCategory": category, "isRetryable": retryable, "message": message}
    payload.update(extra)
    return {"content": [{"type": "text", "text": json.dumps(payload, indent=2)}],
            "is_error": True}

@tool(
    "get_customer",
    "Verify who you are talking to, from an email address, a phone number or a customer "
    "id. Returns a verified customer id only when exactly one account matches. When more "
    "than one matches it returns every candidate and no verified id, so ask the customer "
    "for another identifier rather than choosing one. Call this before any order or "
    "refund tool.",
    {"identifier": str},
)
async def get_customer(args):
    raw = str(args["identifier"]).strip()
    needle = raw.lower()
    matches = [c for c in load("customers")
               if needle in (c["customer_id"].lower(), c["email"].lower(),
                             c["phone"].lower(), c["name"].lower())]
    if not matches:
        return failure(
            "validation", "No account matches " + raw + ".", True,
            suggestion="Ask for the email address, phone number or an order number.")
    if len(matches) > 1:
        # A successful query with an ambiguous answer. Returning one "most likely"
        # match here would hide the ambiguity instead of resolving it.
        return ok({
            "matches": len(matches),
            "verified_customer_id": None,
            "candidates": [{"customer_id": c["customer_id"], "name": c["name"],
                            "email_hint": mask(c["email"]),
                            "phone_hint": "*******" + c["phone"][-3:]} for c in matches],
            "next_action": "Ask the customer for an email address, phone number or order "
                           "number before taking any customer-specific action.",
        })
    found = dict(matches[0])
    found["matches"] = 1
    found["verified_customer_id"] = found["customer_id"]
    return ok(found)

## 3. Call them directly

**An SDK tool is not an ordinary function.** `@tool` returns an `SdkMcpTool` object, so
`get_customer(...)` raises `TypeError`: it is not callable. The work is on `.handler`, and
`.name`, `.description` and `.input_schema` are exactly what the model is shown.

That is the opposite of Lab 1's FastMCP tools, which stayed ordinary functions you could
call directly, and it is why a failure there had to be raised rather than returned.

Six calls below, covering all five shapes. Two of them are textually identical on purpose:
`ORD-0009` times out the first time and succeeds the second, which is what makes retrying a
transient failure worth doing exactly once.

In [3]:
print(f"callable(get_customer): {callable(T.get_customer)}, "
      f"type: {type(T.get_customer).__name__}\n")


def outcome(payload):
    """The one thing worth reading off each answer."""
    if "message" in payload:
        return payload["message"]
    if "matches" in payload:
        return f"{payload['matches']} matched, verified_customer_id={payload['verified_customer_id']}"
    return f"{payload['order_id']}, {len(payload)} fields"


CASES = [
    ("one match",      T.get_customer,   {"identifier": "ivan.petrov@example.com"}),
    ("two matches",    T.get_customer,   {"identifier": "Sam Okafor"}),
    ("no match",       T.get_customer,   {"identifier": "nobody@example.com"}),
    ("timeout",        T.lookup_order,   {"order_id": "ORD-0009", "customer_id": "CUST-0007"}),
    ("retried",        T.lookup_order,   {"order_id": "ORD-0009", "customer_id": "CUST-0007"}),
    ("outside window", T.process_refund, {"order_id": "ORD-0009", "customer_id": "CUST-0007",
                                          "amount": 120.0, "reason": "faulty zip"}),
]

T.reset()
rows = []
for label, tool, arguments in CASES:
    is_error, payload = await labkit.call_sdk_tool(tool, **arguments)
    rows.append((label, "is_error" if is_error else "ok",
                 payload.get("errorCategory", "-"), outcome(payload)))

labkit.show_table(rows, headers=("case", "flag", "category", "what came back"),
                  wrap={"what came back": 52})

callable(get_customer): False, type: SdkMcpTool

case            flag      category    what came back                                      
--------------  --------  ----------  ----------------------------------------------------
one match       ok        -           1 matched, verified_customer_id=CUST-0001
two matches     ok        -           2 matched, verified_customer_id=None
no match        is_error  validation  No account matches nobody@example.com.
timeout         is_error  transient   Timed out after 5000 ms calling the orders API.
retried         ok        -           ORD-0009, 33 fields
outside window  is_error  business    This order was delivered 67 days ago, which is
                                      outside the 30 day refund window, so a refund cannot
                                      be processed. A replacement or store credit is
                                      available instead, and a human can approve an
                                      exception 

**Read the second row against the third.** Two matches came back as a success with no
verified id; no match came back as a validation failure. The ambiguity is the answer. The
absence is a fault in the input.

## 4. The record is a data problem before it is a reasoning problem

**`lookup_order` returns thirty-three fields. Five of them decide a refund.**

The other twenty-eight are paid for on every turn they stay in the window, and they are read
on every turn too.

Worse, the two tools disagree about how to write things down. `get_customer` returns Unix
timestamps; `lookup_order` returns ISO 8601 and a numeric status code. Ask the model whether
one order is older than another and you have handed it a formats question instead of a
business question. It will usually get it right, and *usually* is the same word that
produced the 12%.

Neither of these is fixed by asking the model to be careful.

In [4]:
_, customer = await labkit.call_sdk_tool(T.get_customer, identifier="CUST-0001")
_, record = await labkit.call_sdk_tool(T.lookup_order, order_id="ORD-0003",
                                       customer_id="CUST-0001")

labkit.show_payload(customer, label="get_customer, the fields carrying a time",
                    keys=("created", "last_contact_at"))
print()
labkit.show_payload(record, label=f"lookup_order, {len(record)} fields, the five a refund turns on",
                    keys=H.KEEP)
print(f"\n  and the other {len(record) - len(H.KEEP)}: "
      + ", ".join(key for key in record if key not in H.KEEP)[:132] + " ...")

get_customer, the fields carrying a time
  created                  1753696964
  last_contact_at          1789812164

lookup_order, 33 fields, the five a refund turns on
  order_id                 ORD-0003
  status                   30
  delivered_at             2026-09-03T10:02:44+00:00
  total_pence              8999
  item_name                Ridgeline walking boots

  and the other 28: customer_id, status_detail, placed_at, dispatched_at, updated_at, currency, subtotal_pence, shipping_pence, tax_pence, discount_penc ...


## 5. One pass fixes both, before the model reads anything

**`PostToolUse` sees a result before the model does, and `updatedToolOutput` replaces it.**

One callback converts the timestamps, labels the status code and drops the twenty-eight
fields nobody asked for, for every tool matching the pattern, including servers whose code
you do not own.

It cannot un-run a call. That is `PreToolUse`'s job, and it is the next two sections.
**Transformation is after; prevention is before.**

One shape note. An MCP tool answers with an array of typed content blocks, so the JSON a
rule needs is inside a text block rather than at the top of the response. `payload_of` in
`support/hooks.py` reads either shape and puts it back the way it found it.

In [5]:
labkit.show_source(H, "normalise_tool_output")

async def normalise_tool_output(input_data, tool_use_id, context):
    """PostToolUse. Unix to ISO 8601, status code to label, thirty-three fields to five."""
    payload, shape = payload_of(input_data.get("tool_response"))
    if payload is None:
        return {}
    normalised = dict(payload)
    changed = False
    for field in ("created", "last_contact_at"):
        value = normalised.get(field)
        if isinstance(value, (int, float)) and not isinstance(value, bool):
            normalised[field] = datetime.fromtimestamp(value, tz=timezone.utc).isoformat()
            changed = True
    status = normalised.get("status")
    if isinstance(status, int) and not isinstance(status, bool):
        normalised["status"] = STATUS_NAMES.get(status, str(status))
        changed = True
    if input_data.get("tool_name") == "mcp__support__lookup_order" and "total_pence" in normalised:
        trimmed = {key: normalised[key] for key in KEEP if key in normalised}
        trimmed["fields_dropped"] = len(normalised) - len(trimmed)
        normalised, changed = trimmed, True
    if not changed:
        return {}
    return {"hookSpecificOutput": {
        "hookEventName": "PostToolUse",
        "updatedToolOutput": repack(input_data["tool_response"], normalised, shape),
    }}

In [7]:
async def through(hook, tool_name, response):
    """Run a PostToolUse hook over a real answer and read what the model would see."""
    updated = labkit.updated_output(
        await labkit.fire(hook, tool_name, tool_response=response))
    return json.loads(updated[0]["text"])


raw = await T.lookup_order.handler({"order_id": "ORD-0003", "customer_id": "CUST-0001"})
before = json.loads(raw["content"][0]["text"])
after = await through(H.normalise_tool_output, "mcp__support__lookup_order", raw["content"])

print(f"lookup_order  {len(before)} fields in, {len(after) - 1} kept, "
      f"{after['fields_dropped']} dropped")
print(f"  status      {before['status']!r}  ->  {after['status']!r}")

raw = await T.get_customer.handler({"identifier": "CUST-0001"})
was = json.loads(raw["content"][0]["text"])["created"]
customer = await through(H.normalise_tool_output, "mcp__support__get_customer", raw["content"])
print(f"get_customer  created     {was!r}  ->  {customer['created']!r}")

lookup_order  33 fields in, 5 kept, 28 dropped
  status      30  ->  'delivered'
get_customer  created     1753696964  ->  '2025-07-28T10:02:44+00:00'


### The case facts, which are the other half of the same question

`support/hooks.py` carries the normaliser **and** the case record, because both answer one
question: what does the harness remember between calls?

The facts block sits outside the summarised history and goes into every turn, one record per
issue, with identifiers and amounts exactly as the tools reported them. A summary would turn
`40.00` into "about forty pounds", and the numbers are what decide which order gets refunded.

In [8]:
H.case["verified_customer_id"] = customer["verified_customer_id"]
H.note_issue("billing", "invoice INV-0002", "40.00", "charged twice, open")
H.note_issue("delivery", "order ORD-0003   SHP-0004", "", "marked delivered, open")

print(H.render_facts())

=== CASE FACTS ===
CUSTOMER  CUST-0001  verified
ISSUE 1   billing
  invoice INV-0002   40.00
  status  charged twice, open
ISSUE 2   delivery
  order ORD-0003   SHP-0004
  status  marked delivered, open
===


## 6. The gate that makes ordering a property of the harness

**The rule is "verify the customer before any order or refund operation". Written in the
system prompt it holds most of the time. Written as two hooks it holds every time.**

`PostToolUse` records a verified id, and only when exactly one account matched. `PreToolUse`
denies the downstream tools until that record exists. Three things make this the answer
rather than a workaround:

- **Ordering stops depending on the model's compliance**, which means it also holds on the 12%.
- **The denial teaches.** `permissionDecisionReason` comes back to the agent as the tool
  result, so it reads why it was stopped and calls `get_customer` next. The workflow
  self-corrects rather than erroring out.
- **It cannot be bypassed.** Hooks are evaluated before the permission mode, so the denial
  stands even under `bypassPermissions`. This is why the gate is a hook and not a mode: we
  measured a mode letting an unlisted tool through, with no denial recorded at all.

Watch what the two-match case does. It records nothing, so the gate stays shut and the agent
has to ask for another identifier. **The ambiguity rule and the ordering rule turn out to be
the same mechanism.**

In [9]:
labkit.show_source(H, "record_verification", "require_verification")

async def record_verification(input_data, tool_use_id, context):
    """PostToolUse. Remember the verified id, and only on exactly one match.

    Two candidates record nothing, so the gate below stays shut and the agent has to
    ask for another identifier. The ambiguity rule and the ordering rule turn out to be
    the same mechanism.
    """
    if input_data.get("tool_name") != "mcp__support__get_customer":
        return {}
    payload, _ = payload_of(input_data.get("tool_response"))
    if payload and payload.get("matches") == 1 and payload.get("verified_customer_id"):
        case["verified_customer_id"] = payload["verified_customer_id"]
        case["customer_name"] = payload.get("name")
    return {}

async def require_verification(input_data, tool_use_id, context):
    """PreToolUse. Deny the downstream tools until the prerequisite has returned.

    The reason comes back to the agent as the tool result, so it reads why it was
    stopped and calls get_customer next. A denial that teaches self-corrects; a bare
    refusal makes the agent guess.
    """
    if input_data.get("tool_name") in GATED and not case["verified_customer_id"]:
        return {"hookSpecificOutput": {
            "hookEventName": "PreToolUse",
            "permissionDecision": "deny",
            "permissionDecisionReason": (
                "Verify the customer with get_customer before any order or refund "
                "operation. If more than one account matched, ask the customer for "
                "another identifier first, then verify."),
        }}
    return {}

In [10]:
REFUND = {"order_id": "ORD-0003", "customer_id": "CUST-0001", "amount": 89.99,
          "reason": "damaged on arrival"}


async def verdict(**tool_input):
    """What the gate says about a refund, without one being made."""
    seen = labkit.decision(
        await labkit.fire(H.require_verification, "mcp__support__process_refund",
                          tool_input=tool_input))
    return "allowed" if seen is None else f"DENIED: {seen[1][:58]}"


async def verify(identifier):
    """Let the PostToolUse hook see a real get_customer answer."""
    answer = await T.get_customer.handler({"identifier": identifier})
    await labkit.fire(H.record_verification, "mcp__support__get_customer",
                      tool_response=answer["content"])
    return H.case["verified_customer_id"]


H.reset()
print(f"before any verification   recorded {str(H.case['verified_customer_id']):<10} "
      f"{await verdict(**REFUND)}")
print(f"after two candidates      recorded {str(await verify('Sam Okafor')):<10} "
      f"{await verdict(**REFUND)}")
print(f"after exactly one         recorded {str(await verify('ivan.petrov@example.com')):<10} "
      f"{await verdict(**REFUND)}")

before any verification   recorded None       DENIED: Verify the customer with get_customer before any order or 
after two candidates      recorded None       DENIED: Verify the customer with get_customer before any order or 
after exactly one         recorded CUST-0001  allowed


## 7. The threshold, and why a denial names its alternative

**Policy says an agent may refund up to 500 pounds. `process_refund` does not enforce that,
on purpose.**

The tool executes what it is told; the limit lives in the harness, where it applies to every
caller and can change without redeploying the backend. Part B watches a 650 pound refund go
through when this hook is not loaded.

**The whole lesson is in the reason string.** A bare refusal leaves the agent guessing, and a
guessing agent either retries the same call or tells the customer nothing useful. Naming
`escalate_to_human` is what turns a refusal into a redirect: the policy violation becomes a
different workflow instead of a dead end.

A hook returning `{}` is how it says: no opinion, carry on.

In [11]:
labkit.show_source(H, "enforce_refund_limit")

async def enforce_refund_limit(input_data, tool_use_id, context):
    """PreToolUse. Block, and redirect by naming the workflow that can proceed."""
    if input_data.get("tool_name") != "mcp__support__process_refund":
        return {}
    try:
        amount = float(input_data.get("tool_input", {}).get("amount") or 0)
    except (TypeError, ValueError):
        amount = 0.0
    if amount > REFUND_LIMIT:
        return {"hookSpecificOutput": {
            "hookEventName": "PreToolUse",
            "permissionDecision": "deny",
            "permissionDecisionReason": (
                "A refund of " + format(amount, ".2f") + " pounds is over the "
                + format(REFUND_LIMIT, ".0f") + " pound agent limit and needs a human. "
                "Call escalate_to_human with the customer id, the root cause, the "
                "amount and a recommended action."),
        }}
    return {}

In [12]:
for amount in (89.99, 500.00, 650.00):
    seen = labkit.decision(
        await labkit.fire(H.enforce_refund_limit, "mcp__support__process_refund",
                          tool_input={**REFUND, "amount": amount}))
    print(f"  {amount:7.2f}  {'allowed' if seen is None else seen[0]}")
    for sentence in (seen[1].split(". ") if seen else []):
        print(f"           {sentence.strip()}")

    89.99  allowed
   500.00  allowed
   650.00  deny
           A refund of 650.00 pounds is over the 500 pound agent limit and needs a human
           Call escalate_to_human with the customer id, the root cause, the amount and a recommended action.


## 8. A handoff the receiving human can act on

**Escalation crosses a context boundary: the human who picks the case up cannot read this
conversation.** They see only what the agent hands over, so the handoff is not a summary of
a chat. It is a record that stands alone.

The exam's minimum set is four fields: customer id, root cause, refund amount, recommended
action. The guide's fuller version adds the issue summary, the order, what was already tried
and which trigger fired. None of them contradict, and all of them earn their place:
`escalation_reason` routes the case to a queue, `refund_amount` decides who has the
authority to approve it, and `actions_taken` stops a human re-offering the replacement the
customer has already refused.

**Structure is what makes this checkable.** A schema can refuse a handoff that forgot the
amount. A paragraph cannot be checked for the field it left out, and it cannot be checked
for pointing at a conversation the reader does not have.

In [13]:
CASES = {
    "a prose summary": {
        "customer_id": "CUST-0001",
        "root_cause": "The customer is unhappy, as discussed above, and wants a refund.",
        "refund_amount": "650.00",
        "recommended_action": "Please take a look and sort it out for them.",
    },
    "missing the amount": {
        "customer_id": "CUST-0001",
        "root_cause": "Espresso machine arrived with a cracked housing; photos supplied.",
        "recommended_action": "Approve the full refund; the fault is not in dispute.",
    },
    "self-contained": {
        "customer_id": "CUST-0001",
        "issue_summary": "Refund request for a damaged espresso machine.",
        "order_id": "ORD-0021",
        "root_cause": "Espresso machine arrived with a cracked housing; photos supplied.",
        "actions_taken": ["Verified the customer with get_customer",
                          "Confirmed order ORD-0021 with lookup_order",
                          "Offered a replacement; the customer declined it"],
        "refund_amount": "650.00",
        "recommended_action": "Approve the full refund of 650.00 pounds.",
        "escalation_reason": "Above the 500 pound agent refund limit.",
    },
}

for label, payload in CASES.items():
    is_error, body = await labkit.call_sdk_tool(T.escalate_to_human, **payload)
    print(f"  {label:20s} {'REFUSED  ' + body['message'] if is_error else 'queued as ' + body['ticket']}")

  a prose summary      REFUSED  The handoff is not self-contained: root_cause.
  missing the amount   REFUSED  The handoff is not self-contained: refund_amount.
  self-contained       queued as ESC-0001


**The first one carries all four required fields and is still refused.** "as discussed
above" points at a transcript the receiving human does not have.

## What part A built

| Diagram box | Where it was built |
|---|---|
| In-process MCP tools, four of them | Section 2 |
| transient retry, business refusal | Sections 2 and 3 |
| PostToolUse normaliser | Section 5 |
| PreToolUse policy, prerequisite | Section 6 |
| PreToolUse policy, threshold | Section 7 |
| Human handoff, self-contained | Section 8 |

**Everything above is deterministic, and that is the point to carry out of this part.** None
of it asked a model for anything, so none of it has a failure rate. The gates hold on every
run, including the run where the model would have got it wrong.

What is left is the half that cannot be guaranteed: whether to resolve or escalate, and
whether to ask or assume. Part B adds the agent, runs six conversations at it, and reports
that half honestly, as a rate.